In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
import datetime

In [17]:
existing_spark = SparkSession.getActiveSession()

if existing_spark:
    print("Existing Spark session found. Stopping it...")
    existing_spark.stop()

Existing Spark session found. Stopping it...


In [5]:
spark = (
        SparkSession.builder
        .appName("Catalog App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    )

26/04/26 10:34:23 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
spark

In [7]:
print("Catalog:", spark.conf.get("spark.sql.catalogImplementation"))

Catalog: hive


##### Set Warehouse dir location either S3 or local path

In [8]:
print("Warehouse Dir:", spark.conf.get("spark.sql.warehouse.dir"))

Warehouse Dir: s3a://shivchoudhury-datasets/warehouse


In [9]:
print("Current Database:", spark.catalog.currentDatabase())

Current Database: default


In [12]:
spark.sql("CREATE DATABASE IF NOT EXISTS glue_catalog.my_glue_db")

26/04/26 10:35:19 WARN ProfileFileReader: Ignoring earlier-seen '[default]', because '[profile default]' was found on line 5


DataFrame[]

In [14]:
spark.sql("USE glue_catalog.my_glue_db")

DataFrame[]

In [15]:
print("Current Database:", spark.catalog.currentDatabase())

Current Database: my_glue_db


In [16]:
spark.catalog.listTables("my_glue_db")

[]

#### Customers Table

In [9]:
# -----------------------
# 2. Customers Data
# -----------------------
customers_data = [
    (1, "Alice Johnson", "alice@example.com", "New York", "2021-01-10"),
    (2, "Bob Smith", "bob@example.com", "Chicago", "2021-02-15"),
    (3, "Charlie Brown", "charlie@example.com", "Los Angeles", "2021-03-20"),
    (4, "David Wilson", "david@example.com", "San Francisco", "2021-04-25"),
    (5, "Eva Davis", "eva@example.com", "Houston", "2021-05-30"),
    (6, "Frank Miller", "frank@example.com", "Miami", "2021-06-10"),
    (7, "Grace Lee", "grace@example.com", "Boston", "2021-07-15"),
    (8, "Hannah Scott", "hannah@example.com", "Seattle", "2021-08-20"),
    (9, "Ian Wright", "ian@example.com", "Denver", "2021-09-25"),
    (10, "Julia Turner", "julia@example.com", "Phoenix", "2021-10-30"),
    (11, "Kevin Harris", "kevin@example.com", "Dallas", "2021-11-05"),
    (12, "Laura King", "laura@example.com", "Austin", "2021-12-10"),
    (13, "Mike Adams", "mike@example.com", "Atlanta", "2022-01-15"),
    (14, "Nina Perez", "nina@example.com", "Orlando", "2022-02-20"),
    (15, "Oscar Clark", "oscar@example.com", "Detroit", "2022-03-25"),
    (16, "Paula Lewis", "paula@example.com", "Columbus", "2022-04-30"),
    (17, "Quinn Hall", "quinn@example.com", "Las Vegas", "2022-05-15"),
    (18, "Rachel Allen", "rachel@example.com", "Philadelphia", "2022-06-20"),
    (19, "Steve Young", "steve@example.com", "San Diego", "2022-07-25"),
    (20, "Tina Evans", "tina@example.com", "Portland", "2022-08-30")
]

customers_df = spark.createDataFrame(customers_data, 
    ["customer_id", "name", "email", "city", "join_date"])

customers_df.write.mode("overwrite").saveAsTable("customers")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
spark.catalog.listTables("my_glue_db")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[Table(name='customers', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='orders', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='people', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='products', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False)]

#### Products Table

In [11]:
# -----------------------
# 3. Products Data
# -----------------------
products_data = [
    (1, "Laptop", "Electronics", 850.00, 50),
    (2, "Smartphone", "Electronics", 600.00, 120),
    (3, "Tablet", "Electronics", 300.00, 80),
    (4, "Headphones", "Accessories", 50.00, 200),
    (5, "Keyboard", "Accessories", 30.00, 150),
    (6, "Mouse", "Accessories", 20.00, 170),
    (7, "Monitor", "Electronics", 200.00, 60),
    (8, "Charger", "Accessories", 25.00, 100),
    (9, "Backpack", "Bags", 40.00, 90),
    (10, "Shoes", "Fashion", 70.00, 70),
    (11, "Jacket", "Fashion", 120.00, 40),
    (12, "Watch", "Fashion", 200.00, 30),
    (13, "Desk", "Furniture", 150.00, 25),
    (14, "Chair", "Furniture", 90.00, 35),
    (15, "Lamp", "Furniture", 45.00, 45),
    (16, "Book", "Stationery", 15.00, 300),
    (17, "Pen", "Stationery", 2.00, 500),
    (18, "Notebook", "Stationery", 5.00, 400),
    (19, "Bottle", "Kitchen", 12.00, 250),
    (20, "Mug", "Kitchen", 8.00, 150)
]

products_df = spark.createDataFrame(products_data, 
    ["product_id", "product_name", "category", "price", "stock"])

products_df.write.mode("overwrite").saveAsTable("products")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
spark.catalog.listTables("my_glue_db")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[Table(name='customers', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='orders', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='people', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='products', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False)]

#### Orders Table

In [13]:
# -----------------------
# 4. Orders Data
# -----------------------
orders_data = [
    (1, 1, 2, 1, "2022-01-05"),
    (2, 2, 5, 2, "2022-01-10"),
    (3, 3, 1, 1, "2022-01-15"),
    (4, 4, 4, 3, "2022-01-20"),
    (5, 5, 3, 1, "2022-01-25"),
    (6, 6, 6, 2, "2022-02-05"),
    (7, 7, 7, 1, "2022-02-10"),
    (8, 8, 8, 1, "2022-02-15"),
    (9, 9, 9, 2, "2022-02-20"),
    (10, 10, 10, 1, "2022-02-25"),
    (11, 11, 12, 1, "2022-03-05"),
    (12, 12, 14, 2, "2022-03-10"),
    (13, 13, 13, 1, "2022-03-15"),
    (14, 14, 11, 1, "2022-03-20"),
    (15, 15, 15, 2, "2022-03-25"),
    (16, 16, 16, 5, "2022-04-05"),
    (17, 17, 18, 3, "2022-04-10"),
    (18, 18, 19, 2, "2022-04-15"),
    (19, 19, 20, 4, "2022-04-20"),
    (20, 20, 17, 10, "2022-04-25")
]

orders_df = spark.createDataFrame(orders_data, 
    ["order_id", "customer_id", "product_id", "quantity", "order_date"])

orders_df.write.mode("overwrite").saveAsTable("orders")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
spark.catalog.listTables("my_glue_db")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[Table(name='customers', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='orders', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='people', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='products', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False)]

#### People Table

In [15]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS my_glue_db.people (
        id INT,
        name STRING,
        age INT
    )
    ROW FORMAT DELIMITED
    FIELDS TERMINATED BY ','
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [16]:
spark.sql("""
    INSERT INTO my_glue_db.people VALUES
    (1, 'Alice', 30),
    (2, 'Bob', 28),
    (3, 'Charlie', 35),
    (4, 'David', 40),
    (5, 'Eva', 25)
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [17]:
spark.catalog.listTables("my_glue_db")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[Table(name='customers', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='orders', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='people', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False), Table(name='products', database='my_glue_db', description=None, tableType='MANAGED', isTemporary=False)]

In [18]:
spark.sql("select * from my_glue_db.people").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 28|
|  3|Charlie| 35|
|  4|  David| 40|
|  5|    Eva| 25|
|  1|  Alice| 30|
|  2|    Bob| 28|
|  3|Charlie| 35|
|  4|  David| 40|
|  5|    Eva| 25|
+---+-------+---+